# Research Decision Framework

## Objective

This notebook converts the validated covariance research into an
explicit portfolio decision framework.

The objective is not to select a methodology solely according to
historical return.

Instead, the decision considers:

- risk-adjusted performance;
- downside risk;
- portfolio turnover;
- implementation characteristics;
- robustness across specifications; and
- research limitations.

The final recommendation must therefore be conditional on the
investment objective and the evidence generated by CARL.

In [1]:
# We reconstruct the validated evidence from the artifacts created

from pathlib import Path
import pandas as pd

evidence_dir = Path(
    "outputs/research_validation"
)

evidence_matrix = pd.read_csv(
    evidence_dir / "evidence_matrix.csv"
)

research_validation = pd.read_csv(
    evidence_dir / "research_validation.csv"
)

training_window_evidence = pd.read_csv(
    evidence_dir / "training_window_evidence.csv"
)

ranking_table = pd.read_csv(
    evidence_dir / "ranking_stability.csv"
)

shrinkage_tradeoff = pd.read_csv(
    evidence_dir / "shrinkage_tradeoff.csv"
)

In [2]:
research_validation

,Dimension,Result
0,Research question,Covariance methodology affects GMV portfolio o...
1,Out-of-sample design,Strict walk-forward evaluation.
2,Covariance methods,"Sample, shrinkage, Ledoit-Wolf."
3,Training-window sensitivity,"Ranking remained stable across 126, 252, and 5..."
4,Shrinkage sensitivity,Performance declined progressively as shrinkag...
5,Performance leader,Sample covariance.
6,Drawdown leader,Ledoit-Wolf among the tested baseline methods.
7,Turnover consideration,Shrinkage reduced turnover relative to sample ...
8,Generalization,Conclusions remain conditional on the tested r...


## Decision Criteria

The covariance methodology will be evaluated using five criteria.

### 1. Risk-adjusted performance

Primary measures:

- Sharpe ratio
- Sortino ratio
- Calmar ratio

### 2. Absolute performance

Supporting measures:

- total return
- annualized return

### 3. Downside risk

Primary measure:

- maximum drawdown

### 4. Implementation characteristics

Primary measure:

- average turnover

### 5. Robustness

The preferred methodology should not depend entirely on one
training-window specification.

## Decision Philosophy

No single metric determines the final decision.

A methodology with the highest return but substantially worse
implementation characteristics should not automatically be treated
as superior.

Similarly, a methodology with the lowest drawdown should not
automatically be preferred if it materially sacrifices
risk-adjusted performance.

The decision therefore evaluates the covariance estimators as
different points on a performance-risk-implementation trade-off
rather than attempting to identify a universally optimal method.

In [3]:
import pandas as pd

from crypto_alpha_lab.data import load_prices
from crypto_alpha_lab.evaluation.covariance_comparison import (
    compare_covariance_methods,
)
from crypto_alpha_lab.research.covariance_experiment import (
    run_covariance_experiment,
)

In [4]:
assets = {
    "BTC": "BTC-USD",
    "ETH": "ETH-USD",
    "SOL": "SOL-USD",
}

close_prices = pd.concat(
    {
        name: load_prices(
            ticker,
            "2021-01-01",
            "2025-12-31",
            refresh=False,
        )["Close"]
        for name, ticker in assets.items()
    },
    axis=1,
).dropna()

close_prices

,BTC,ETH,SOL
Date,,,
2021-01-01,29374.152344,730.367554,1.842084
2021-01-02,32127.267578,774.534973,1.799275
2021-01-03,32782.023438,975.507690,2.161752
2021-01-04,31971.914062,1040.233032,2.485097
2021-01-05,33992.429688,1100.006104,2.157217
...,...,...,...
2025-12-26,87301.429688,2925.745605,122.195953
2025-12-27,87802.156250,2947.998291,124.647713
2025-12-28,87835.835938,2948.568115,125.199356


In [5]:
experiments = [
    run_covariance_experiment(
        prices=close_prices,
        method="sample",
        train_size=252,
        test_size=21,
    ),
    run_covariance_experiment(
        prices=close_prices,
        method="shrinkage",
        train_size=252,
        test_size=21,
        shrinkage=0.25,
    ),
    run_covariance_experiment(
        prices=close_prices,
        method="ledoit_wolf",
        train_size=252,
        test_size=21,
    ),
]

In [6]:
[(e.method, len(e.returns), len(e.folds)) for e in experiments]

[('sample', 1554, 74), ('shrinkage', 1554, 74), ('ledoit_wolf', 1554, 74)]

In [7]:
from crypto_alpha_lab.evaluation.covariance_comparison import (
    compare_covariance_methods,
)

comparison = compare_covariance_methods(
    experiments=experiments,
    periods_per_year=365,
    risk_free_rate=0.0,
)


In [8]:
decision_table = (
    comparison.summary[
        [
            "total_return",
            "annualized_return",
            "annualized_volatility",
            "sharpe_ratio",
            "sortino_ratio",
            "maximum_drawdown",
            "calmar_ratio",
            "average_turnover",
        ]
    ]
    .copy()
)

decision_table

,total_return,annualized_return,annualized_volatility,sharpe_ratio,sortino_ratio,maximum_drawdown,calmar_ratio,average_turnover
method,,,,,,,,
sample,0.869704,0.158333,0.525500,0.542223,0.041746,-0.781959,0.213557,1.219442
shrinkage,0.781683,0.145287,0.523977,0.520786,0.039903,-0.767388,0.201527,1.110687
ledoit_wolf,0.568794,0.111563,0.525942,0.464014,0.035355,-0.725831,0.168007,1.352342


In [9]:
leaders = pd.Series(
    {
        "total_return": decision_table[
            "total_return"
        ].idxmax(),

        "annualized_return": decision_table[
            "annualized_return"
        ].idxmax(),

        "sharpe_ratio": decision_table[
            "sharpe_ratio"
        ].idxmax(),

        "sortino_ratio": decision_table[
            "sortino_ratio"
        ].idxmax(),

        "maximum_drawdown": decision_table[
            "maximum_drawdown"
        ].idxmax(),

        "calmar_ratio": decision_table[
            "calmar_ratio"
        ].idxmax(),

        "average_turnover": decision_table[
            "average_turnover"
        ].idxmin(),
    },
    name="leader",
)

leaders

total_return              sample
annualized_return         sample
sharpe_ratio              sample
sortino_ratio             sample
maximum_drawdown     ledoit_wolf
calmar_ratio              sample
average_turnover       shrinkage
Name: leader, dtype: str

## 6A.7 — Multi-Objective Research Decision

The baseline comparison does not identify a single covariance
methodology that dominates every evaluation criterion.

Instead, the results reveal three distinct leadership profiles:

- **Sample covariance** leads on return and risk-adjusted performance.
- **Ledoit-Wolf covariance** leads on maximum-drawdown protection.
- **Explicit shrinkage** leads on portfolio turnover.

This indicates that covariance-method selection is inherently
multi-objective.

The appropriate methodology therefore depends on the relative
importance assigned to performance, downside protection, and
implementation stability.

In [10]:
objective_framework = pd.DataFrame(
    {
        "objective": [
            "Return",
            "Risk-adjusted performance",
            "Downside protection",
            "Implementation stability",
        ],
        "metric": [
            "total_return",
            "sharpe_ratio",
            "maximum_drawdown",
            "average_turnover",
        ],
        "preferred_method": [
            "sample",
            "sample",
            "ledoit_wolf",
            "shrinkage",
        ],
    }
)

objective_framework

,objective,metric,preferred_method
0,Return,total_return,sample
1,Risk-adjusted performance,sharpe_ratio,sample
2,Downside protection,maximum_drawdown,ledoit_wolf
3,Implementation stability,average_turnover,shrinkage


In [11]:
sample = decision_table.loc["sample"]
shrinkage = decision_table.loc["shrinkage"]
ledoit_wolf = decision_table.loc["ledoit_wolf"]

performance_comparison = pd.DataFrame(
    {
        "sample_vs_shrinkage": {
            "total_return": (
                sample["total_return"]
                - shrinkage["total_return"]
            ),
            "sharpe_ratio": (
                sample["sharpe_ratio"]
                - shrinkage["sharpe_ratio"]
            ),
            "maximum_drawdown": (
                sample["maximum_drawdown"]
                - shrinkage["maximum_drawdown"]
            ),
            "turnover": (
                sample["average_turnover"]
                - shrinkage["average_turnover"]
            ),
        },
        "sample_vs_ledoit_wolf": {
            "total_return": (
                sample["total_return"]
                - ledoit_wolf["total_return"]
            ),
            "sharpe_ratio": (
                sample["sharpe_ratio"]
                - ledoit_wolf["sharpe_ratio"]
            ),
            "maximum_drawdown": (
                sample["maximum_drawdown"]
                - ledoit_wolf["maximum_drawdown"]
            ),
            "turnover": (
                sample["average_turnover"]
                - ledoit_wolf["average_turnover"]
            ),
        },
    }
)

performance_comparison

,sample_vs_shrinkage,sample_vs_ledoit_wolf
total_return,0.088021,0.300910
sharpe_ratio,0.021437,0.078208
maximum_drawdown,-0.014571,-0.056128
turnover,0.108755,-0.132900


In [12]:
# Decision Score
# we'll demonstrate how the decision changes depending on the objective.
# We'll do this by creating a normalized score

score_data = decision_table[
    [
        "total_return",
        "sharpe_ratio",
        "maximum_drawdown",
        "average_turnover",
    ]
].copy()

score_data


,total_return,sharpe_ratio,maximum_drawdown,average_turnover
method,,,,
sample,0.869704,0.542223,-0.781959,1.219442
shrinkage,0.781683,0.520786,-0.767388,1.110687
ledoit_wolf,0.568794,0.464014,-0.725831,1.352342


In [13]:
#Then normalize the metrics.
# For return, Sharpe and drawdown:

benefit_metrics = [
    "total_return",
    "sharpe_ratio",
    "maximum_drawdown",
]

cost_metrics = [
    "average_turnover",
]

scores = pd.DataFrame(
    index=score_data.index
)

for metric in benefit_metrics:
    minimum = score_data[metric].min()
    maximum = score_data[metric].max()

    scores[metric] = (
        score_data[metric] - minimum
    ) / (
        maximum - minimum
    )

for metric in cost_metrics:
    minimum = score_data[metric].min()
    maximum = score_data[metric].max()

    scores[metric] = (
        maximum - score_data[metric]
    ) / (
        maximum - minimum
    )

scores

,total_return,sharpe_ratio,maximum_drawdown,average_turnover
method,,,,
sample,1.000000,1.000000,0.000000,0.549958
shrinkage,0.707486,0.725904,0.259606,1.000000
ledoit_wolf,0.000000,0.000000,1.000000,0.000000


### Weighting Principle

A composite score requires assumptions about the relative importance
of return, risk-adjusted performance, downside protection, and
turnover.

Those weights are investment-policy assumptions rather than purely
empirical facts.

Therefore, CARL does not select arbitrary weights and present the
resulting score as an objective truth.

Instead, the framework evaluates alternative objective priorities.

In [14]:
# Scenario-Based Decision Analysis. We'll define three reasonable investor objectives.

decision_scenarios = {
    "performance_focused": {
        "total_return": 0.30,
        "sharpe_ratio": 0.40,
        "maximum_drawdown": 0.20,
        "average_turnover": 0.10,
    },
    "risk_control_focused": {
        "total_return": 0.15,
        "sharpe_ratio": 0.25,
        "maximum_drawdown": 0.50,
        "average_turnover": 0.10,
    },
    "implementation_focused": {
        "total_return": 0.15,
        "sharpe_ratio": 0.20,
        "maximum_drawdown": 0.15,
        "average_turnover": 0.50,
    },
}



In [15]:
scenario_scores = {}

for scenario, weights in decision_scenarios.items():

    scenario_scores[scenario] = (
        scores[list(weights)]
        .mul(
            pd.Series(weights)
        )
        .sum(axis=1)
    )

scenario_scores = pd.DataFrame(
    scenario_scores
)

scenario_scores

,performance_focused,risk_control_focused,implementation_focused
method,,,
sample,0.754996,0.454996,0.624979
shrinkage,0.654528,0.517402,0.790245
ledoit_wolf,0.200000,0.500000,0.150000


In [16]:
scenario_winners = scenario_scores.idxmax()

scenario_winners

performance_focused          sample
risk_control_focused      shrinkage
implementation_focused    shrinkage
dtype: str

## Scenario-Based Interpretation

The scenario analysis demonstrates an important property of the
research:

> The preferred covariance methodology depends on the objective
> function.

A performance-focused investor places greater emphasis on return and
risk-adjusted performance.

A risk-control-focused investor assigns greater importance to
drawdown protection.

An implementation-focused investor assigns greater importance to
turnover.

This prevents CARL from reducing a multi-dimensional portfolio
decision to a single historical performance ranking.

# Research Decision

## Baseline recommendation

For the current CARL research specification, **sample covariance is
the preferred baseline estimator**.

The evidence supporting this choice is:

1. highest total return;
2. highest annualized return;
3. highest Sharpe ratio;
4. highest Sortino ratio;
5. highest Calmar ratio; and
6. first-place Sharpe ranking across all tested training windows.

However, this recommendation is conditional.

### Alternative objectives

**Ledoit-Wolf** is preferable when downside protection receives
greater importance because it produced the least severe maximum
drawdown in the baseline comparison.

**Explicit shrinkage** is preferable when implementation stability
receives greater importance because it produced the lowest turnover.

### Final research position

The evidence therefore supports sample covariance as the **baseline
performance-oriented specification**, while retaining shrinkage and
Ledoit-Wolf as economically meaningful alternatives.

No methodology is declared universally superior.

## Scenario-Based Research Conclusion

The scenario analysis demonstrates that covariance-method selection is
objective-dependent.

Sample covariance achieves the highest composite score under the
performance-focused objective (0.7550), consistent with its leadership
in total return, annualized return, Sharpe ratio, Sortino ratio, and
Calmar ratio.

Shrinkage achieves the highest score under both the risk-control-focused
objective (0.5174) and the implementation-focused objective (0.7902).
Its advantage is associated with its lower portfolio turnover and
competitive downside characteristics.

Ledoit-Wolf does not achieve the highest composite score in any of the
three scenarios, although it provides the strongest standalone
maximum-drawdown result in the baseline comparison.

The results therefore do not support a universal covariance winner.
Instead, they support an objective-dependent methodology choice:

- Sample covariance for performance-oriented research.
- Shrinkage for a stronger balance between risk control and
  implementation stability.
- Ledoit-Wolf remains a relevant robustness benchmark because of its
  distinct covariance-estimation methodology and downside behavior.

The composite scores are not universal truths. They depend on the
explicit objective weights selected for each scenario. They are
therefore used as decision-analysis tools rather than as evidence that
one estimator is universally superior.

### Scenario-Based Decision

The scenario analysis shows that covariance-method selection is
objective-dependent.

- **Performance-focused:** Sample covariance is preferred.
- **Risk-control-focused:** Shrinkage is preferred.
- **Implementation-focused:** Shrinkage is preferred.

Therefore, CARL does not identify a universally superior covariance
estimator. Instead, it demonstrates that the appropriate methodology
depends on the research objective and the relative importance assigned
to performance, risk control, and implementation considerations.

This provides a more decision-oriented conclusion than selecting a
single estimator solely on the basis of total return.